<a href="https://colab.research.google.com/github/G0rav/machine_learning_explained_visually_free/blob/main/05-multiple-linear-regression/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multiple Linear Regression

**[Watch the video](https://youtu.be/lS16A11OiME)** · video 5 of
*Machine Learning Explained Visually*.

Everything from the video, runnable. Three layers:

1. **Follow along** — every fit in the video, in code, in the same order.
2. **Experiment** — push each idea past where the video stopped.
3. **Challenge** — write the pieces yourself.

Nothing here imports from anywhere. `numpy` is required; `scikit-learn` is
used in two places and each says so; `matplotlib` is optional. Run it top to
bottom.

Every section asserts the number the video put on screen. If a check ever
fails, the video and this notebook have drifted apart and one of them is wrong.

---

# Layer 1 — Follow along

In [ ]:
import numpy as np

PASSED = 0
def check(label, got, want, tol=1e-4):
    """Assert a value the video states, and say so."""
    global PASSED
    got_f, want_f = float(got), float(want)
    assert abs(got_f - want_f) <= tol, f"{label}: {got_f!r} != {want_f!r}"
    PASSED += 1
    print(f"ok   {label:44s} {got_f:.6g}")

## The cars

The six hero cars are the ones from video 1. The lot is sixty cars from a
generator, rebuilt here from its seed so this file reproduces the video
exactly rather than quoting it.

Three facts are built into the generator, and each earns a beat later:
depreciation is a **percentage** per year rather than an amount, price varies
**more** when the price is higher, and mileage tracks age at about 11,500 a
year — close enough to be nearly redundant, which is the whole of part 3.

In [ ]:
HERO_AGE   = np.array([1.0, 2.0, 4.0, 6.0, 8.0, 9.0])
HERO_PRICE = np.array([23.2, 19.6, 15.2, 11.2, 7.6, 7.2])
HERO_MILEAGE = 12.0 * HERO_AGE          # exactly twelve thousand a year

rng = np.random.default_rng(20260809)
age = np.sort(rng.uniform(0.4, 14.0, 60))
expected_miles = 11.5 * age
mileage = np.maximum(expected_miles + rng.normal(0.0, 14.0, 60), 0.8)
excess = mileage - expected_miles
true_price = 25.0 * np.exp(-0.105 * age) - 0.085 * excess
price = true_price + rng.normal(0.0, 0.09 * np.abs(true_price) + 0.30, 60)

LOT_AGE   = np.round(age, 1)
LOT_MILES = np.round(mileage, 1)
LOT_PRICE = np.round(np.maximum(price, 0.8), 1)

print(f"{len(LOT_AGE)} cars, age {LOT_AGE.min():.1f}-{LOT_AGE.max():.1f} years")

In [ ]:
def design(*columns):
    """Feature columns with a leading column of ones for the intercept."""
    n = len(np.asarray(columns[0], float))
    return np.column_stack([np.ones(n)] + [np.asarray(c, float) for c in columns])

def fit(X, y):
    """Solve the normal equations. Solve them -- never invert."""
    return np.linalg.solve(X.T @ X, X.T @ y)

def r_squared(X, y, theta):
    resid = y - X @ theta
    return float(1.0 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean())))

## Part 1 — More than one feature

One weight per column, and the intercept is the weight on a column of ones.
Fit age alone first, because that is the number the second column has to beat.

In [ ]:
X1 = design(LOT_AGE)
t1 = fit(X1, LOT_PRICE)
check("age only, slope",     t1[1], -1.2285)
check("age only, intercept", t1[0], 21.4246)
check("age only, R^2", r_squared(X1, LOT_PRICE, t1), 0.8784)

X2 = design(LOT_AGE, LOT_MILES)
t2 = fit(X2, LOT_PRICE)
check("age + mileage, w_age",     t2[1], -0.5043)
check("age + mileage, w_mileage", t2[2], -0.0614)
check("age + mileage, intercept", t2[0], 21.3373)
check("age + mileage, R^2", r_squared(X2, LOT_PRICE, t2), 0.9052)

print(f"\nthe age coefficient moved {t1[1]:.4f} -> {t2[1]:.4f}, "
      f"and nothing about the cars changed")

## Part 2 — Feature standardisation

`-0.5043` against `-0.0614` looks like a factor of eight. It is a factor of
whichever units you wrote the data down in — so record mileage in single miles
and watch the weight move by a thousand while the model does not move at all.

In [ ]:
X_miles = design(LOT_AGE, LOT_MILES * 1000.0)
t_miles = fit(X_miles, LOT_PRICE)
check("in single miles, w_mileage", t_miles[2], -0.00006141, tol=1e-8)
check("in single miles, R^2", r_squared(X_miles, LOT_PRICE, t_miles), 0.9052)
print("same R^2, weight a thousand times smaller: the ranking was about units")

In [ ]:
def standardise(a):
    a = np.asarray(a, float)
    return (a - a.mean()) / a.std(), a.mean(), a.std()

z_age, age_mean, age_sd = standardise(LOT_AGE)
z_mil, mil_mean, mil_sd = standardise(LOT_MILES)

Xz = design(z_age, z_mil)
tz = fit(Xz, LOT_PRICE)
check("standardised w_age",     tz[1], -1.9844)
check("standardised w_mileage", tz[2], -2.9723)
check("standardised intercept", tz[0], 11.7833)
check("standardised R^2", r_squared(Xz, LOT_PRICE, tz), 0.9052)

print(f"\nranking flips: mileage {abs(tz[2]):.4f} now beats age {abs(tz[1]):.4f}")

## Part 3 — Collinearity

On the hero six, mileage is *exactly* twelve times age. So any pair of weights
with `w_age + 12 * w_mileage = -2` predicts identically — an entire line of
models the data cannot choose between.

In [ ]:
def hero_predict(w_age, w_mil, b=24.0):
    return b + w_age * HERO_AGE + w_mil * HERO_MILEAGE

def hero_sse(w_age, w_mil, b=24.0):
    r = HERO_PRICE - hero_predict(w_age, w_mil, b)
    return float(r @ r)

for w_mil in (0.0, -0.05, -0.10, -0.20):
    w_age = -2.0 - 12.0 * w_mil
    preds = hero_predict(w_age, w_mil)
    assert np.allclose(preds, [22, 20, 16, 12, 8, 6])
    check(f"SSE at w_age={w_age:+.2f}", hero_sse(w_age, w_mil), 4.4800)

print("\nfour different stories about cars, one set of predictions")

In [ ]:
corr = float(np.corrcoef(LOT_AGE, LOT_MILES)[0, 1])
check("lot correlation", corr, 0.9588)

### The perturbation test

Shake every standardised value by a whisker, refit, and see whether the answer
survives. If it moves a lot, the features are collinear and the coefficients
are not yours to read.

In [ ]:
def perturbed_fits(n=5, sigma=0.1, seed=7, lam=0.0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        noise = rng.normal(0.0, sigma, (len(z_age), 2))
        X = design(z_age + noise[:, 0], z_mil + noise[:, 1])
        if lam <= 0:
            theta = fit(X, LOT_PRICE)
        else:
            pen = np.eye(X.shape[1]) * lam
            pen[0, 0] = 0.0                      # never the intercept
            theta = np.linalg.solve(X.T @ X + pen, X.T @ LOT_PRICE)
        out.append((theta[1], theta[2]))
    return np.array(out)

runs = perturbed_fits()
swing = runs[:, 0].max() - runs[:, 0].min()
for i, (wa, wm) in enumerate(runs, 1):
    print(f"  run {i}:  w_age {wa:+.4f}   w_mileage {wm:+.4f}")
check("w_age swing over five shakes", swing, 0.8694)
print("\nand every time age gets smaller, mileage gets bigger to compensate")

## Part 4 — Overfitting

Ten columns of coin flips. They cannot say anything about a car, and ordinary
least squares gives every one of them a weight — and the score goes **up**.

In [ ]:
coins = np.random.default_rng(20260811).integers(0, 2, size=(60, 59)).astype(float)

def standardise_cols(A):
    mean, sd = A.mean(axis=0), A.std(axis=0)
    sd[sd == 0] = 1.0
    return (A - mean) / sd, mean, sd

raw12 = np.column_stack([LOT_AGE, LOT_MILES, coins[:, :10]])
Z12, _, _ = standardise_cols(raw12)
X12 = np.column_stack([np.ones(60), Z12])
t12 = np.linalg.lstsq(X12, LOT_PRICE, rcond=None)[0]

print(f"R^2 with two real columns : {r_squared(Xz, LOT_PRICE, tz):.4f}")
print(f"R^2 with ten coins added  : {r_squared(X12, LOT_PRICE, t12):.4f}")
print(f"\nlargest coin weight: {np.abs(t12[3:]).max():.4f}")
print("a new column with weight zero reproduces the old model exactly,")
print("so the best fit with it can never be worse -- and never is")

### Measure it properly

Hold eighteen cars back. Fit on forty-two, score on the eighteen the model has
never seen, and standardise using the **training** mean and sd so the held-out
rows never touch the fit.

In [ ]:
order = np.random.default_rng(20260810).permutation(60)
TRAIN, TEST = np.sort(order[:42]), np.sort(order[42:])

def ladder(k):
    cols = np.column_stack([LOT_AGE, LOT_MILES, coins[:, :k]]) if k else \
           np.column_stack([LOT_AGE, LOT_MILES])
    tr, te = cols[TRAIN], cols[TEST]
    mean, sd = tr.mean(axis=0), tr.std(axis=0)
    sd[sd == 0] = 1.0
    Ztr, Zte = (tr - mean) / sd, (te - mean) / sd
    A = np.column_stack([np.ones(len(Ztr)), Ztr])
    theta = np.linalg.lstsq(A, LOT_PRICE[TRAIN], rcond=None)[0]
    def score(Z, y):
        resid = y - np.column_stack([np.ones(len(Z)), Z]) @ theta
        return float(1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean())))
    w = theta[1:]
    return (score(Ztr, LOT_PRICE[TRAIN]), score(Zte, LOT_PRICE[TEST]),
            float(np.sqrt(w @ w)))

print(f"{'coins':>6}  {'fitted on':>10}  {'held out':>10}  {'||w||':>8}")
for k in (0, 5, 10, 20, 30, 39):
    tr, te, nw = ladder(k)
    print(f"{k:6d}  {tr:10.4f}  {te:10.4f}  {nw:8.2f}")

tr39, te39, nw39 = ladder(39)
print("\nthirty-nine coin flips and forty-two cars: a perfect fit on what it")
print("was shown, and worse than predicting the average on what it was not.")
print("zero is what you score by ignoring the data and guessing the mean.")

## Part 5 — Ridge regression

Memorising needed the weight vector to grow. So make weights expensive:
minimise the squared error **plus** `lambda` times the sum of the squared
weights. That is ridge, and the penalty is an L2 regulariser.

In [ ]:
def ridge(X, y, lam):
    """Penalised normal equations, intercept column left unpenalised."""
    A = np.column_stack([np.ones(len(X)), X])
    pen = np.eye(A.shape[1]) * lam
    pen[0, 0] = 0.0
    return np.linalg.solve(A.T @ A + pen, A.T @ y)

cols39 = np.column_stack([LOT_AGE, LOT_MILES, coins[:, :39]])
tr_raw, te_raw = cols39[TRAIN], cols39[TEST]
mean, sd = tr_raw.mean(axis=0), tr_raw.std(axis=0); sd[sd == 0] = 1.0
Ztr, Zte = (tr_raw - mean) / sd, (te_raw - mean) / sd
ytr, yte = LOT_PRICE[TRAIN], LOT_PRICE[TEST]

def held_out(theta):
    resid = yte - np.column_stack([np.ones(len(Zte)), Zte]) @ theta
    return float(1 - (resid @ resid) / ((yte - yte.mean()) @ (yte - yte.mean())))

plain = np.linalg.lstsq(np.column_stack([np.ones(len(Ztr)), Ztr]), ytr,
                        rcond=None)[0]
penalised = ridge(Ztr, ytr, 5.0)

print(f"no penalty  held out {held_out(plain):+.4f}   "
      f"||w|| {np.sqrt(plain[1:] @ plain[1:]):.2f}")
print(f"lambda = 5  held out {held_out(penalised):+.4f}   "
      f"||w|| {np.sqrt(penalised[1:] @ penalised[1:]):.2f}")
print("\nand the weight vector came back smaller than it was before any")
print("coin flips were added at all")

In [ ]:
ridge_runs = perturbed_fits(lam=10.0)
ridge_swing = ridge_runs[:, 0].max() - ridge_runs[:, 0].min()
check("w_age swing under ridge", ridge_swing, 0.2456)

tz_ridge = ridge(np.column_stack([z_age, z_mil]), LOT_PRICE, 10.0)
print(f"\nand the bill on the honest two-column model: "
      f"R^2 {r_squared(Xz, LOT_PRICE, tz):.4f} -> "
      f"{r_squared(Xz, LOT_PRICE, tz_ridge):.4f}")
print("not a better fit -- a more repeatable one")

## Part 6 — Lasso regression

Swap the squares for absolute values and you get lasso. The whole difference
is one derivative: `d(w^2)/dw = 2w` dies as the weight approaches zero, and
`d|w|/dw = ±1` does not.

In [ ]:
r = 0.01
for w in (0.05, 0.01):
    l2_next = w - r * 2 * w
    l1_next = max(w - r * 1.0, 0.0)
    print(f"from {w:.2f}:  squared penalty -> {l2_next:.4f}   "
          f"absolute value -> {l1_next:.4f}")

check("L2 from 0.05", 0.05 - r * 2 * 0.05, 0.049)
check("L1 from 0.05", 0.05 - r,            0.040)
check("L2 from 0.01", 0.01 - r * 2 * 0.01, 0.0098)
check("L1 from 0.01", max(0.01 - r, 0.0),  0.0)
print("\nthe squared penalty stalls; the absolute value arrives")

In [ ]:
from sklearn.linear_model import Lasso, Ridge   # the only library fits here

lasso = Lasso(alpha=0.50, max_iter=200000).fit(Z12, LOT_PRICE)
ridge12 = Ridge(alpha=10.0).fit(Z12, LOT_PRICE)

survivors = int((np.abs(lasso.coef_) > 1e-9).sum())
print(f"lasso, lambda 0.50 : {survivors} of 12 weights survive")
print(f"   age {lasso.coef_[0]:+.4f}   mileage {lasso.coef_[1]:+.4f}")
print(f"ridge, lambda 10   : {int((ridge12.coef_ == 0).sum())} of 12 are zero, "
      f"smallest |w| {np.abs(ridge12.coef_).min():.6f}")

check("lasso survivors", survivors, 2, tol=0)
check("lasso w_age",     lasso.coef_[0], -1.7320)
check("lasso w_mileage", lasso.coef_[1], -2.7143)
print("\nit found the two real columns without being told which they were")

## Part 7 — Outliers

A price is one keystroke from being wrong by a factor of ten. This car sold
for 4.5 and was recorded as 45.0.

In [ ]:
TYPO_AGE, TYPO_PRICE = 13.4, 45.0
age_bad   = np.append(LOT_AGE, TYPO_AGE)
price_bad = np.append(LOT_PRICE, TYPO_PRICE)

Xb = design(age_bad)
tb = fit(Xb, price_bad)
check("with the bad row, slope", tb[1], -1.0006)
check("with the bad row, R^2", r_squared(Xb, price_bad, tb), 0.3582)

resid = price_bad - Xb @ tb
worst = int(np.argmax(np.abs(resid)))
check("its residual", resid[worst], 38.1360)
check("next largest", np.sort(np.abs(resid))[-2], 4.6291)
print(f"\none row in {len(age_bad)}: R^2 {r_squared(X1, LOT_PRICE, t1):.4f}"
      f" -> {r_squared(Xb, price_bad, tb):.4f}")

In [ ]:
keep = np.arange(len(age_bad)) != worst
t_back = fit(design(age_bad[keep]), price_bad[keep])
check("after dropping the worst residual", t_back[1], -1.2285)
print("fit, look at the residuals, drop what is absurd, fit again")
print("-- that idea is the core of RANSAC, and it works well beyond linear models")

In [ ]:
def line_with_bad_row_at(age_at):
    a = np.append(LOT_AGE, age_at)
    p = np.append(LOT_PRICE, TYPO_PRICE)
    return fit(design(a), p)

print(f"the same wrong price at 13.4 years: slope "
      f"{line_with_bad_row_at(13.4)[1]:.4f}")
print(f"the same wrong price at  5.0 years: slope "
      f"{line_with_bad_row_at(5.0)[1]:.4f}")
print("\nsame error, much less damage -- because of where it sat")

---

# Layer 2 — Experiment

Past where the video stopped.

### 1. Drop mileage, and watch the age coefficient come home

The fastest way to feel that a coefficient is not a fact about the world.

In [ ]:
print(f"with mileage in the model : {fit(X2, LOT_PRICE)[1]:.4f}")
print(f"with mileage taken out    : {fit(X1, LOT_PRICE)[1]:.4f}")

### 2. Dial the correlation and watch the swing

Build two columns whose correlation you choose, and run the perturbation test
on each. The collinearity is the cause; the unstable coefficients are only the
symptom.

In [ ]:
def swing_at(rho, seed=0, n=400):
    g = np.random.default_rng(seed)
    a = g.normal(size=n)
    b = rho * a + np.sqrt(max(1 - rho ** 2, 0.0)) * g.normal(size=n)
    y = 2.0 * a + 1.0 * b + g.normal(0, 0.5, n)
    out = []
    for _ in range(12):
        noise = g.normal(0, 0.1, (n, 2))
        out.append(fit(design(a + noise[:, 0], b + noise[:, 1]), y)[1])
    return max(out) - min(out)

for rho in (0.0, 0.5, 0.9, 0.99, 0.999):
    print(f"correlation {rho:<6} -> w1 swings {swing_at(rho):.4f}")

### 3. The whole ridge path

The video showed six values of lambda. Here is the curve, including the fact
that the intercept never moves.

In [ ]:
Z2 = np.column_stack([z_age, z_mil])
print(f"{'lambda':>8}  {'w_age':>9}  {'w_mileage':>10}  {'intercept':>10}  {'R^2':>7}")
for lam in (0, 1, 10, 50, 100, 500):
    th = ridge(Z2, LOT_PRICE, lam)
    print(f"{lam:8}  {th[1]:9.4f}  {th[2]:10.4f}  {th[0]:10.4f}  "
          f"{r_squared(Xz, LOT_PRICE, th):7.4f}")

### 4. The lasso path, and where each coin dies

Find the lambda at which the first junk column is switched off, and the one at
which only the two real columns are left.

In [ ]:
for lam in (0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00):
    w = Lasso(alpha=lam, max_iter=200000).fit(Z12, LOT_PRICE).coef_
    alive = [i for i in range(12) if abs(w[i]) > 1e-9]
    names = ["age", "mileage"] + [f"coin{i+1}" for i in range(10)]
    print(f"lambda {lam:<5} -> {len(alive):2d} survive: "
          f"{', '.join(names[i] for i in alive) if len(alive) <= 6 else '...'}")

---

# Layer 3 — Challenge

Write these before you look at the answers above.

### 1. Soft thresholding

Implement lasso with coordinate descent. The exact zeros should come out of
the threshold, not out of a rounding rule. Ten lines is enough.

In [ ]:
def my_lasso(X, y, lam, iters=2000):
    """Return (weights, intercept). Zeros must be exactly zero."""
    # your code here
    raise NotImplementedError

# w, b = my_lasso(Z12, LOT_PRICE, 0.50)
# assert int((np.abs(w) > 1e-9).sum()) == 2

### 2. The perturbation test, from scratch

Four lines: fit, shake, refit, compare. Then run it on a dataset you trust and
one you do not, and see whether it tells them apart.

In [ ]:
def my_perturbation_test(X_columns, y, sigma=0.1, runs=5):
    """Return the swing of each coefficient across the runs."""
    # your code here
    raise NotImplementedError

### 3. Predict before you run it

Add a third feature that is pure noise to the two real columns. Before you fit:
does `R^2` on the fitted rows go up, down, or stay the same? Does `R^2` on
held-out rows? Write your answer down, then check.

In [ ]:
# your prediction here, then:
# noise_col = np.random.default_rng(0).normal(size=60)
# ...

### 4. Find the bad row without being told

Implement fit → drop the worst residual → refit, repeated until nothing is
absurd any more. Then plant a corrupt row somewhere you choose and see whether
your loop finds it — and whether where you put it changes the answer.

In [ ]:
def drop_outliers_and_refit(ages, prices, threshold=4.0):
    """Return (slope, intercept, dropped_indices)."""
    # your code here
    raise NotImplementedError

In [ ]:
print(f"\n{PASSED} checks passed.")

---

**What is still open:** nothing here told you how to choose `lambda`. Every
number above was measured on the data it was fitted to, except the held-out
block in parts 4 and 5 — and choosing `lambda` properly means measuring on
data the model has never seen, systematically. That is its own subject.

**[Watch the video](https://youtu.be/lS16A11OiME)**